### **Step 1: Mount google Drive**

In [1]:
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install jsonlines

In [4]:
import pandas as pd
import jsonlines

### **Step -2 : Load Data**
#### Function to read .jsonl file into a DataFrame

In [5]:
def load_jsonlines_to_dataframe(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for obj in reader:
            data.append(obj)
    return pd.DataFrame(data)

#### Load the datasets

In [6]:
us_train_path = '/content/drive/MyDrive/us_train_data_final_OFFICIAL.jsonl'

In [7]:
us_train = load_jsonlines_to_dataframe(us_train_path)

In [8]:
us_train.head()

,bill_id,text,summary,title,text_len,sum_len
0,107_hr2256,SECTION 1. SHORT TITLE.\n\n This Act may be...,Border Hospital Survival and Illegal Immigrant...,To amend the Public Health Service Act to esta...,6100,527
1,111_hr4710,SECTION 1. SHORT TITLE.\n\n This Act may be...,Farm to School Improvements Act of 2010 - Amen...,To amend the Richard B. Russell National Schoo...,8628,1161
2,107_s409,SECTION 1. SHORT TITLE.\n\n This Act may be...,Persian Gulf War Illness Compensation Act of 2...,"A bill to amend title 38, United States Code, ...",5567,694
3,109_s2759,SECTION 1. SHORT TITLE.\n\n This Act may be...,Medicare Part D Outreach and Enrollment Enhanc...,A bill to provide for additional outreach and ...,6361,810
4,107_hr5568,SECTION 1. SHORT TITLE.\n\n This Act may be...,Seniors' Retirement Recovery Act of 2002 - Ame...,To amend the Internal Revenue Code of 1986 to ...,5368,380


In [9]:
us_test_path = '/content/drive/MyDrive/us_test_data_final_OFFICIAL.jsonl'

In [10]:
us_test = load_jsonlines_to_dataframe(us_test_path)

In [11]:
us_test.head()

,bill_id,text,summary,title,text_len,sum_len
0,110_hr37,SECTION 1. SHORT TITLE.\n\n This Act may be...,National Science Education Tax Incentive for B...,To amend the Internal Revenue Code of 1986 to ...,8494,321
1,112_hr2873,SECTION 1. SHORT TITLE.\n\n This Act may be...,Small Business Expansion and Hiring Act of 201...,To amend the Internal Revenue Code of 1986 to ...,6522,1424
2,109_s2408,SECTION 1. RELEASE OF DOCUMENTS CAPTURED IN IR...,Requires the Director of National Intelligence...,A bill to require the Director of National Int...,6154,463
3,108_s1899,SECTION 1. SHORT TITLE.\n\n This Act may be...,National Cancer Act of 2003 - Amends the Publi...,A bill to improve data collection and dissemin...,19853,1400
4,107_s1531,SECTION 1. SHORT TITLE.\n\n This Act may be...,Military Call-up Relief Act - Amends the Inter...,A bill to amend the Internal Revenue Code of 1...,6273,278


In [12]:
ca_test_data = '/content/drive/MyDrive/ca_test_data_final_OFFICIAL.jsonl'

In [13]:
ca_test = load_jsonlines_to_dataframe(ca_test_data)

In [14]:
ca_test.head()

,bill_id,text,summary,title,sum_len,text_len
0,SB 2,The people of the State of California do enact...,Existing property tax law establishes a vetera...,An act to amend Section 215.1 of the Revenue a...,1181,8203
1,SB 6,The people of the State of California do enact...,Existing law provides that the Board of Parole...,"An act to amend Section 3550 of, and to add Se...",1435,8975
2,SB 8,The people of the State of California do enact...,The Sales and Use Tax Law imposes a tax on ret...,An act\nto add Chapter 3.8 (commencing with Se...,1170,13667
3,SB 9,The people of the State of California do enact...,"Existing law requires all moneys, except for f...","An act to amend Sections 75220, 75221, and 752...",3050,11091
4,SB 19,The people of the State of California do enact...,Existing law defines a request regarding resus...,An act to add and repeal Section 4788 of the P...,3255,6624


### **Step -3 : Preprocess Data**

In [15]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [16]:
from datasets import Dataset

In [17]:
def preprocess_data(example):
    return {
        "input_text": f"summarize:{example['text']}",
        "target_text": example["summary"]
    }

#### Convert training data to Hugging Face Dataset format

In [18]:
train_data = Dataset.from_pandas(us_train)
train_data = train_data.map(preprocess_data, remove_columns = ['bill_id', 'text', 'summary', 'title', 'text_len', 'sum_len'])

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

### **Step -4 : Tokenization**

In [19]:
def tokenize_data(batch):
    input_encodings = tokenizer(batch['input_text'], padding="max_length", truncation=True, max_length=512)
    target_encodings = tokenizer(batch['target_text'], padding="max_length", truncation=True, max_length=150)

    return {
        "input_ids": input_encodings['input_ids'],
        "attention_mask": input_encodings['attention_mask'],
        "label": target_encodings['input_ids'],
    }

In [20]:
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
)

#### Initialize tokenizer and model

In [21]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [22]:
train_data = train_data.map(tokenize_data, batched=True)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

### **Step -5 : Model Training**

In [23]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [24]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_steps=10_000,
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=train_data,
    tokenizer=tokenizer,
)

trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-24-83e7e80e6a08>:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,4.053900,1.808247
2,3.894900,1.741764
3,3.804500,1.725188


TrainOutput(global_step=7107, training_loss=4.0170656169066365, metrics={'train_runtime': 2221.316, 'train_samples_per_second': 25.592, 'train_steps_per_second': 3.199, 'total_flos': 7693775388278784.0, 'train_loss': 4.0170656169066365, 'epoch': 3.0})

### **Step -6: Evaluation**

In [25]:
def generate_summary(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [26]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.4 MB/s eta 0:00:00


In [27]:
import evaluate

In [28]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=88dd4d46e541a81951ad252a99a2c30b16aa9b9f1187bd97040bf65b0a71dd23
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


#### Evaluate on test data

In [29]:
rouge = evaluate.load("rouge")

In [30]:
def generate_summary(input_text):
    device = model.device  # Ensure input is moved to the same device as the model
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [31]:
def evaluate_model(dataset):
    device = model.device  # Use the same device as the model
    model.to(device)  # Ensure the model is on the right device

    summaries = []
    references = []

    for _, row in dataset.iterrows():
        # Move inputs to the same device
        input_text = f"summarize: {row['text']}"
        generated = generate_summary(input_text)
        summaries.append(generated)
        references.append(row["summary"])

    # Compute Rouge scores
    results = rouge.compute(predictions=summaries, references=references)
    print(results)

In [32]:
model = T5ForConditionalGeneration.from_pretrained("t5-small").to("cuda")

#### Example evaluation on ca_test

In [33]:
evaluate_model(ca_test)

{'rouge1': 0.24490434931957902, 'rouge2': 0.1049475374391366, 'rougeL': 0.16856659615957298, 'rougeLsum': 0.16879539430682172}


In [51]:
model.save_pretrained("/content/t5_model")
tokenizer.save_pretrained("/content/t5_tokenizer")

('/content/t5_tokenizer/tokenizer_config.json',
 '/content/t5_tokenizer/special_tokens_map.json',
 '/content/t5_tokenizer/spiece.model',
 '/content/t5_tokenizer/added_tokens.json')

In [85]:
def summarize_text(input_text):

    # Preprocess the input text
    input_text = f"summarize: {input_text}"

    # Tokenize and encode the input text
    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    # Generate the summary
    summary_ids = model.generate(
    input_ids,
    max_length=300,  # Shorter max length
    min_length=100,  # Ensures minimum content
    length_penalty=1.2,  # Moderate penalty for long texts
    num_beams=3,  # Balanced diversity
    early_stopping=True
    )
    # Decode and return the summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [86]:
input_text = "ABC Corporation announced its Q3 FY2024 financial results, showcasing robust performance across critical metrics. The company reported a revenue of ₹15,000 crore, reflecting a 12% year-over-year growth fueled by strong demand in the technology and services sectors. Net profit for the quarter rose to ₹2,500 crore, a 20% increase compared to ₹2,100 crore in Q3 FY2023, driven by enhanced operational efficiency and cost optimization initiatives. The earnings per share (EPS) also improved, reaching ₹15.5 from ₹13 in the prior year. Significant achievements during the quarter included the expansion into three new international markets, contributing ₹500 crore to overall revenue, and the successful launch of the ABC Cloud Platform, which generated ₹1,000 crore in revenue within its first quarter. Additionally, operational expenses were reduced by 15% compared to the same period last year. Looking ahead, the company projected Q4 FY2024 revenues to be between ₹16,000 crore and ₹17,000 crore, supported by seasonal demand and the sustained growth of its cloud business."

summary = summarize_text(input_text)
print("Original Text:")
print(input_text)
print("\nGenerated Summary:")
print(summary)

Original Text:
ABC Corporation announced its Q3 FY2024 financial results, showcasing robust performance across critical metrics. The company reported a revenue of ₹15,000 crore, reflecting a 12% year-over-year growth fueled by strong demand in the technology and services sectors. Net profit for the quarter rose to ₹2,500 crore, a 20% increase compared to ₹2,100 crore in Q3 FY2023, driven by enhanced operational efficiency and cost optimization initiatives. The earnings per share (EPS) also improved, reaching ₹15.5 from ₹13 in the prior year. Significant achievements during the quarter included the expansion into three new international markets, contributing ₹500 crore to overall revenue, and the successful launch of the ABC Cloud Platform, which generated ₹1,000 crore in revenue within its first quarter. Additionally, operational expenses were reduced by 15% compared to the same period last year. Looking ahead, the company projected Q4 FY2024 revenues to be between ₹16,000 crore and ₹1